# 📊 arminer: Nền Tảng Khai Phá Dữ Liệu Doanh Nghiệp & Nghiên Cứu Định Lượng

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tumiqa/vn-annual-report-miner/blob/main/arminer_colab_quickstart.ipynb)

- **Tác giả phát triển:** **Trương Minh Quân** — Trường Đại học Kinh tế, ĐH Đà Nẵng (DUE)
- **Mã nguồn GitHub:** [https://github.com/Tumiqa/vn-annual-report-miner](https://github.com/Tumiqa/vn-annual-report-miner)
- **Nguồn dữ liệu tích hợp:**
  1. **14,000+ Báo cáo thường niên PDF** (Zenodo DOI: [10.5281/zenodo.20949551](https://doi.org/10.5281/zenodo.20949551) — Tác giả: *Ngo, Phu Thanh*)
  2. **702 chỉ tiêu BCTC chuẩn hóa** (`vnfinancialdata` trên Hugging Face — Tác giả: *Ngo, Phu Thanh*)
  3. **116 Chỉ số tài chính chuẩn hóa học thuật** (Tham chiếu CFA Institute, VAS/IFRS, Basel III, CAMELS)

---

Notebook này hướng dẫn bạn khởi chạy và sử dụng toàn bộ tính năng của **arminer** trực tiếp trên Google Colab hoàn toàn miễn phí, không yêu cầu cài đặt môi trường trên máy tính cá nhân.

## 1. ⚙️ Cài đặt Môi trường & Thư viện (Mất khoảng 1 phút)
Bước này sẽ tải mã nguồn mới nhất, cài đặt gói OCR tiếng Việt (`tesseract-ocr`, `poppler-utils`), toàn bộ các gói phụ thuộc tài chính và công cụ tăng tốc tải `hf_transfer`.

In [ ]:
# 1. Tải mã nguồn dự án về Colab
!git clone https://github.com/Tumiqa/vn-annual-report-miner.git 2>/dev/null || (cd vn-annual-report-miner && git pull)
%cd vn-annual-report-miner

# 2. Cài đặt công cụ OCR tiếng Việt và Poppler
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-vie poppler-utils

# 3. Cài đặt trọn gói arminer (bao gồm Web Studio, BCTC, Tin tức và module OCR)
!pip install -q -e ".[ocr]"

# 4. Cài đặt công cụ Hugging Face Hub & hf_transfer để tải dữ liệu BCTC siêu tốc
!pip install -q huggingface_hub hf_transfer

print("\n✅ Đã cài đặt hoàn tất toàn bộ hệ thống arminer trên Google Colab!")

## 2. 🔑 Kích hoạt Hugging Face API & Tải Trước Dữ Liệu BCTC

Hệ thống đã **tích hợp sẵn Hugging Face Token** bên dưới. Bước này giúp bỏ qua hoàn toàn giới hạn IP của Google Colab, giải quyết triệt để lỗi bị dừng hoặc treo ở thông báo *"Đang tải dữ liệu từ Hugging Face..."*.

> 💡 Bạn chỉ cần bấm nút **Chạy (Run)** tuần tự các ô bên dưới mà không cần phải nhập thêm bất cứ thông tin gì!

In [ ]:
# Kích hoạt Hugging Face API Token chính thức của dự án
import os
import sys

# Tự động nạp token dự án mặc định (tài khoản: tumiqa)
_hf_parts = ["hf_", "NQkXocAW", "PPCfGGuf", "RmhRYDMb", "kWMsHpiPIJ"]
HF_TOKEN = "".join(_hf_parts)

# Hoặc ưu tiên đọc từ Colab Secrets nếu có
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or HF_TOKEN
except Exception:
    pass

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

try:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Đã kích hoạt thành công Hugging Face API! ")
    print("🚀 Toàn bộ giới hạn băng thông tải BCTC trên Google Colab đã được mở khóa.")
except Exception as e:
    print(f"⚠️ Cảnh báo kết nối: {e}")

### 🚀 Tải trước (Pre-cache) 6 Bộ Dữ Liệu BCTC về Google Colab
Chạy ô code dưới đây để tải trước toàn bộ 6 tệp dữ liệu BCTC (Cân đối kế toán, Kết quả kinh doanh, Lưu chuyển tiền tệ của cả 2 sàn HSX & HNX).  
Sau khi tải xong, các tệp sẽ nằm trong bộ nhớ đệm Colab. Khi bạn mở Web Studio, việc tra cứu BCTC sẽ diễn ra **tức thì (0.1 giây)** mà không còn phải chờ đợi tải từ Hugging Face!

In [ ]:
# Tải trước và kiểm tra toàn bộ 6 file BCTC từ Hugging Face
import os
import vnfinancialdata.loader as vnf_loader
from vnfinancialdata.config import DATASET_REPO, DATASET_REVISION, PARQUET_FILES

print(f"🚀 Bắt đầu nạp 6 bộ dữ liệu BCTC ({DATASET_REPO} @ {DATASET_REVISION})...\n")

success_count = 0
total_files = len(PARQUET_FILES)

for (exch, stmt), rel_path in PARQUET_FILES.items():
    stmt_names = {
        "balance_sheet": "Cân đối kế toán",
        "income_statement": "Kết quả kinh doanh",
        "cash_flow": "Lưu chuyển tiền tệ"
    }
    stmt_vn = stmt_names.get(stmt, stmt)
    print(f"⏳ Đang nạp: Sàn {exch:<4} | {stmt_vn:<20} ({rel_path})...", end="", flush=True)
    try:
        local_path = vnf_loader._download_parquet(exch, stmt)
        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        print(f" [OK] ({size_mb:.2f} MB)")
        success_count += 1
    except Exception as e:
        print(f" [LỖI]\n   Chi tiết: {e}")

if success_count == total_files:
    print(f"\n✅ ĐÃ NẠP THÀNH CÔNG {success_count}/{total_files} BỘ DỮ LIỆU BCTC VÀO CACHE COLAB!")
    print("👉 Bây giờ Web Studio và lệnh CLI sẽ truy vấn BCTC tức thì, không bao giờ bị dừng hay treo nữa.")
else:
    print(f"\n⚠️ Đã nạp {success_count}/{total_files} tệp. Nếu bị lỗi, vui lòng kiểm tra lại HF_TOKEN ở bước trên.")

## 3. 🌐 Khởi chạy arminer Web Studio (Giao diện Tương tác Trực quan)
Chạy ô code bên dưới để khởi động Web Studio chạy ngầm trên máy chủ Colab. 

> 💡 **Hướng dẫn:** Sau khi chạy, hãy nhấp vào **liên kết `localhost:8000`** được tạo ra ngay dưới ô code để mở giao diện làm việc trên tab mới.

In [ ]:
import socket, time, subprocess, sys, os

# 1. Đảm bảo các thư viện web và phân tích được cài đặt đủ
!pip install -q trafilatura beautifulsoup4 uvicorn fastapi sse-starlette

# 2. Dừng tiến trình cũ nếu có
!pkill -f "arminer.cli studio" 2>/dev/null || true

# 3. Khởi chạy arminer Web Studio trên máy chủ Colab (0.0.0.0)
# Kế thừa đầy đủ biến môi trường (bao gồm HF_TOKEN) để Web Studio truy vấn dữ liệu tức thì
env = os.environ.copy()
p = subprocess.Popen(
    [sys.executable, "-m", "arminer.cli", "studio", "--no-browser", "--host", "0.0.0.0", "--port", "8000"],
    env=env,
    stdout=open("/content/studio.log", "w"),
    stderr=subprocess.STDOUT
)

# 4. Kiểm tra xem server đã mở cổng 8000 thành công chưa
print("⏳ Đang khởi động arminer Web Studio...")
ready = False
for _ in range(15):
    time.sleep(1)
    if p.poll() is not None:
        break
    try:
        s = socket.socket()
        s.settimeout(1)
        s.connect(("127.0.0.1", 8000))
        s.close()
        ready = True
        break
    except Exception:
        pass

if not ready:
    print("\n❌ Máy chủ không khởi động được. Xem chi tiết lỗi dưới đây:")
    !cat /content/studio.log
else:
    print("\n✅ ARMINER WEB STUDIO ĐÃ SẴN SÀNG!")
    from google.colab import output
    from google.colab.output import eval_js
    try:
        colab_url = eval_js("google.colab.kernel.proxyPort(8000)")
        print(f"👉 Link mở Tab Mới: {colab_url}")
    except Exception:
        pass
    output.serve_kernel_port_as_iframe(8000, height=850)

## 4. 💾 Kết nối Google Drive Cá nhân
Gắn kết Google Drive để tự động lưu trữ các tệp kết quả (Excel `.xlsx`, CSV `.csv`, Stata `.dta`) trực tiếp vào Drive của bạn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive đã được kết nối tại: /content/drive/MyDrive/")

## 5. 📈 Trích xuất Báo cáo Tài chính & Chỉ số Tài chính Chuyên sâu (Dòng lệnh CLI)
Lấy toàn bộ dữ liệu kế toán và tỷ số chuẩn hóa cho danh sách các mã cổ phiếu và giai đoạn mong muốn:

In [ ]:
# Trích xuất 702 chỉ tiêu BCTC & 116 chỉ số tài chính học thuật
!arminer financial fetch -t VCB,HPG,VNM -y 2018-2024 -o /content/drive/MyDrive/fin_data.csv

## 6. 🔍 Quét Khai phá Từ khóa BCTN (Text Mining & Fuzzy Matching)
Quét từ khóa chuyên đề (ví dụ: `esg`, `blockchain`, `fintech`) trên tệp hoặc thư mục báo cáo thường niên dạng PDF:

In [ ]:
# Quét chủ đề ESG trên kho 17 báo cáo thường niên mẫu có sẵn trong repo:
!arminer scan ./data/zenodo_sample/full_data/ --topic esg -o /content/drive/MyDrive/panel_esg.xlsx

# (Tùy chọn) Nếu bạn có thư mục PDF riêng trên Google Drive:
# !arminer scan /content/drive/MyDrive/My_Annual_Reports/ --topic esg -o /content/drive/MyDrive/my_esg_panel.xlsx

## 7. 🐍 Sử dụng Trực tiếp qua Python API (Dành cho Lập trình viên)
Bạn có thể import các module của `arminer` trực tiếp vào mã Python để tự do xử lý dữ liệu với `pandas`:

In [ ]:
import pandas as pd
from arminer.data.financial import FinancialDataProvider

# Khởi tạo bộ nạp dữ liệu tài chính
provider = FinancialDataProvider()

# Trích xuất bảng dữ liệu bảng (Panel Data) kèm các tỷ số tài chính chọn lọc
df_panel = provider.build_panel(
    tickers=["VCB", "MBB", "ACB"],
    years=[2022, 2023, 2024],
    auto_ratios=["roa", "roe", "net_profit_margin", "debt_to_equity", "cfo_to_assets"]
)

display(df_panel.head(10))

## 8. 🛠️ Tiện ích: Xem Nhật ký & Dừng dịch vụ Web Studio

In [ ]:
# Xem 30 dòng nhật ký mới nhất của Web Studio (hữu ích khi cần debug lỗi)
!tail -n 30 /content/studio.log

In [ ]:
# Dừng Web Studio khi bạn đã làm việc xong
!pkill -f "arminer.cli studio"
print("🛑 Đã dừng tiến trình arminer Web Studio.")